# Phase 2: Context-Aware Toxicity Detection (RoBERTa Fine-Tuning)

In this notebook, we fine-tune a pre-trained `roberta-base` model on our dataset. RoBERTa (Robustly Optimized BERT Approach) is highly effective for context-heavy NLP tasks.

**Objective:** Teach the model to distinguish between genuine toxicity and domain-specific gaming jargon (e.g., "kill", "shoot") by adapting its contextual embeddings to our specific dataset.

*Note: For local testing, we are using a smaller subset of the data. Full training should be conducted on a GPU environment.*

In [1]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from transformers import Trainer, TrainingArguments
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Check for available device (GPU if available, otherwise CPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cpu


## 1. Data Preparation and Tokenization
We load the dataset and initialize the `RobertaTokenizer`. We must convert our raw text into input IDs and attention masks that the Transformer model can process.

In [2]:
print("Loading data...")
df = pd.read_csv('../data/train.csv')[['comment_text', 'toxic']].dropna()

# LOCAL TESTING: We sample 2000 rows for now since the full dataset might overload the CPU.
# IMPORTANT: Remove the ".sample(n=2000, ...)" part when moving to a GPU environment for full training.
df_sampled = df.sample(n=2000, random_state=42)

X_train, X_val, y_train, y_val = train_test_split(
    df_sampled['comment_text'].tolist(), 
    df_sampled['toxic'].tolist(), 
    test_size=0.2, 
    random_state=42
)

print("Loading Tokenizer...")
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

print("Tokenizing datasets...")
train_encodings = tokenizer(X_train, truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(X_val, truncation=True, padding=True, max_length=128)

Loading data...
Loading Tokenizer...
Tokenizing datasets...


## 2. Creating PyTorch Datasets
Hugging Face `Trainer` requires the data to be in a specific PyTorch Dataset format. We create a custom class for this purpose.

In [3]:
class ToxicityDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        # Convert encodings to PyTorch tensors
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# Instantiate the dataset objects
train_dataset = ToxicityDataset(train_encodings, y_train)
val_dataset = ToxicityDataset(val_encodings, y_val)

## 3. Metrics and Model Initialization
Since our dataset suffers from severe class imbalance, accuracy is not a reliable metric. We must explicitly track Precision, Recall, and the F1-Score during training.

In [4]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    
    # Calculate Precision, Recall, and F1-Score for the binary classification task
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

print("Loading RoBERTa Model...")
# Initialize the model with 2 output labels (Toxic or Non-Toxic)
model = RobertaForSequenceClassification.from_pretrained('roberta-base', num_labels=2)
model.to(device)

Loading RoBERTa Model...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
  

## 4. Training
We configure the `TrainingArguments` and start the fine-tuning process. We will train for just 1 epoch locally to verify the pipeline.

In [5]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=1,              # Train for only 1 epoch for local testing
    per_device_train_batch_size=8,   # Small batch size to avoid overloading the CPU
    per_device_eval_batch_size=8,    # Small batch size for evaluation
    warmup_steps=50,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=50,
    eval_strategy="steps",           
    eval_steps=50,
    save_strategy="no"               # Do not save weights locally to save disk space
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

print("Starting training (This will take a few minutes on CPU)...")
trainer.train()

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Starting training (This will take a few minutes on CPU)...


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
50,0.389839,0.360947,0.925000,0.000000,0.000000,0.000000
100,0.334727,0.192391,0.960000,0.692308,0.818182,0.600000
150,0.213315,0.142588,0.957500,0.721311,0.709677,0.733333
200,0.176219,0.142479,0.965000,0.787879,0.722222,0.866667


TrainOutput(global_step=200, training_loss=0.27852497100830076, metrics={'train_runtime': 643.9501, 'train_samples_per_second': 2.485, 'train_steps_per_second': 0.311, 'total_flos': 105244422144000.0, 'train_loss': 0.27852497100830076, 'epoch': 1.0})